# Solutions - Assignment 1: Gathering data by downloading a file

## Instructor reference

This notebook is a complete worked solution to [`topic19_assignment_file_download.ipynb`](../../data_analysis_process/assignments/topic19_assignment_file_download.ipynb).

**It solves Option B, the Our World in Data life expectancy dataset**, comparing Ghana, Kenya, and Senegal from 1960 to 2023. Option B was chosen for this reference because it contains the most instructive cleaning problem: the `Entity` column mixes real countries with regional aggregates such as `Africa` and `World`, and those aggregate rows are identifiable only because their `Code` is missing.

A student who chose Option A or C should have a structurally identical notebook with different column names. Mark the structure and the reasoning, not the resemblance to this file.

### Marking notes are inline

Cells beginning **"Marking note"** flag what to look for and the mistakes that recur. Read those before marking a batch.

### Running this notebook

It downloads about 590 KB from `ourworldindata.org` and needs an internet connection on first run. The `requests` sections are guarded, so it still runs where `requests` is not installed.

## Part 1 - Set up your workspace

Unchanged from the assignment. Nothing for the student to do here beyond running it.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import hashlib
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
CHART_DIR = BASE_DIR / "charts"

for folder in (RAW_DIR, CLEAN_DIR, CHART_DIR):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

USER_AGENT = "ITFY-Ghana-student-assignment/1.0"

print("pandas version:     ", pd.__version__)
print("requests available: ", REQUESTS_AVAILABLE)
print("raw data folder:    ", RAW_DIR)
print("clean data folder:  ", CLEAN_DIR)

## Part 2 - Choose your dataset and write your question

**TODOs 1 to 5.**

In [ ]:
DATASET_OPTIONS = {
    "A": {
        "name": "Gapminder five-year country indicators",
        "url": "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv",
        "raw_filename": "gapminder_five_year_raw.csv",
        "publisher": "Gapminder Foundation (via the Plotly datasets collection)",
        "documentation": "https://www.gapminder.org/data/documentation/",
        "row_meaning": "one country in one year",
        "country_column": "country", "year_column": "year",
        "value_column": "lifeExp", "value_unit": "years",
    },
    "B": {
        "name": "Our World in Data life expectancy",
        "url": "https://ourworldindata.org/grapher/life-expectancy.csv",
        "raw_filename": "owid_life_expectancy_raw.csv",
        "publisher": "Our World in Data",
        "documentation": "https://ourworldindata.org/life-expectancy",
        "row_meaning": "one entity in one year",
        "country_column": "Entity", "year_column": "Year",
        "value_column": "Life expectancy", "value_unit": "years",
    },
    "C": {
        "name": "World Bank GDP by country and year",
        "url": "https://raw.githubusercontent.com/datasets/gdp/main/data/gdp.csv",
        "raw_filename": "world_bank_gdp_raw.csv",
        "publisher": "World Bank (redistributed by Frictionless Data / datahub.io)",
        "documentation": "https://datahub.io/core/gdp",
        "row_meaning": "one country in one year",
        "country_column": "Country Name", "year_column": "Year",
        "value_column": "Value", "value_unit": "current US dollars",
    },
}

# TODO 1
CHOSEN_OPTION = "B"

# TODO 2
RESEARCH_QUESTION = ("How did life expectancy at birth change in Ghana between 1960 "
                     "and 2023, compared with Kenya and Senegal?")

# TODO 3 - spellings confirmed against the data in Part 5 before being fixed here.
FOCUS_COUNTRIES = ["Ghana", "Kenya", "Senegal"]

# TODO 4 - 1960 is the start of reliable annual estimates for these three
# countries; 2023 is the last year in the file.
START_YEAR = 1960
END_YEAR = 2023

# TODO 5
EXPECTATION = ("I expect all three to have improved substantially, because child "
               "mortality fell across West and East Africa over this period, and I "
               "expect a visible dip in Kenya around 2000 because of the effect of "
               "HIV/AIDS on adult mortality.")

if CHOSEN_OPTION not in DATASET_OPTIONS:
    raise ValueError('Set CHOSEN_OPTION to "A", "B", or "C" before continuing.')

DATASET = DATASET_OPTIONS[CHOSEN_OPTION]
SOURCE_URL = DATASET["url"]
RAW_PATH = RAW_DIR / DATASET["raw_filename"]

print("Dataset:  ", DATASET["name"])
print("URL:      ", SOURCE_URL)
print("Raw file: ", RAW_PATH)
print("Question: ", RESEARCH_QUESTION)
print("Countries:", FOCUS_COUNTRIES)
print("Years:    ", START_YEAR, "to", END_YEAR)

> **Marking note - Part 2.** The question must be answerable with the two required charts. Reject "What is life expectancy?" and anything with no comparison or no time dimension. `EXPECTATION` is not marked for being correct - it is marked for existing and being reasoned. A student who predicted the Kenya dip and then found it has done real analytical thinking; so has one who predicted something that turned out to be wrong and said so honestly in Part 12.

## Part 3 - Download the file, two ways

**TODOs 6 to 15.**

The worked example cell from the assignment is omitted here; it was already complete.

In [ ]:
def download_with_urllib(url, destination, force=False, timeout=30):
    """Download a file using the standard library and return provenance details."""
    destination = Path(destination)

    # TODO 6 - use the cached copy unless force=True.
    if destination.exists() and not force:
        print(f"Using cached file: {destination.name}")
        return {
            "tool": "urllib",
            "url": url,
            "path": str(destination),
            "bytes": destination.stat().st_size,
            "status": None,          # no request was made
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "from_cache": True,
        }

    # TODO 7 - send the request, distinguishing the two failure modes.
    request = Request(url, headers={"User-Agent": USER_AGENT})
    try:
        with urlopen(request, timeout=timeout) as response:
            status = response.status
            content = response.read()
    except HTTPError as error:
        raise RuntimeError(
            f"The server refused {url} with HTTP status {error.code}"
        ) from error
    except URLError as error:
        raise RuntimeError(
            f"Could not reach the server for {url}: {error.reason}"
        ) from error

    # TODO 8 - write the bytes exactly as received.
    destination.write_bytes(content)
    print(f"Downloaded {len(content):,} bytes to {destination.name}")

    # TODO 9
    return {
        "tool": "urllib",
        "url": url,
        "path": str(destination),
        "bytes": len(content),
        "status": status,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "from_cache": False,
    }


def download_with_requests(url, destination, force=False, timeout=30):
    """Download a file using requests and return the same dictionary shape."""
    if not REQUESTS_AVAILABLE:
        raise RuntimeError("requests is not installed. Run: pip install requests")

    destination = Path(destination)

    # TODO 10
    if destination.exists() and not force:
        print(f"Using cached file: {destination.name}")
        return {
            "tool": "requests",
            "url": url,
            "path": str(destination),
            "bytes": destination.stat().st_size,
            "status": None,
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "from_cache": True,
        }

    # TODO 11 - raise_for_status turns a bad status into an exception.
    response = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=timeout)
    response.raise_for_status()

    # TODO 12
    destination.write_bytes(response.content)
    print(f"Downloaded {len(response.content):,} bytes to {destination.name}")
    return {
        "tool": "requests",
        "url": url,
        "path": str(destination),
        "bytes": len(response.content),
        "status": response.status_code,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "from_cache": False,
    }

In [ ]:
# TODO 13
urllib_provenance = download_with_urllib(SOURCE_URL, RAW_PATH)
print(json.dumps(urllib_provenance, indent=2))

# TODO 14
requests_path = RAW_DIR / ("requests_" + DATASET["raw_filename"])
if REQUESTS_AVAILABLE:
    requests_provenance = download_with_requests(SOURCE_URL, requests_path)
    print(json.dumps(requests_provenance, indent=2))
else:
    requests_provenance = None
    print("requests not installed - skipping the second download.")

# TODO 15 - the tool cannot change the bytes, so the two files must match.
if requests_provenance is not None:
    same_size = urllib_provenance["bytes"] == requests_provenance["bytes"]
    same_bytes = RAW_PATH.read_bytes() == requests_path.read_bytes()
    print("\nSame byte count: ", same_size)
    print("Byte-for-byte identical:", same_bytes)

> **Marking note - Part 3.** The four things that most often go wrong:
>
> 1. **The function prints instead of returning.** Part 4 then fails, because `describe_source()` has nothing to read. This is the single most common failure and it is worth 4 of the 16 marks on its own.
> 2. **No caching.** A function that re-downloads on every run is impolite to the server and slow. Look for the `destination.exists() and not force` guard.
> 3. **Catching bare `Exception`.** Accept it, but note that `HTTPError` and `URLError` mean genuinely different things - the server refused you, versus you never reached the server. A student who separated them has understood something real.
> 4. **Missing timeout.** Without one, a hung server hangs the notebook indefinitely.
>
> Also check for `write_bytes()` rather than `write_text()`. Writing text forces a decode and can corrupt a file that is not UTF-8; the raw file should be the bytes that arrived.

### Comparing the two tools - model answer

| Question | Answer |
|---|---|
| Which took more lines, and why? | `urllib`. It needs a `Request` object for the header, a context manager, an explicit `.read()`, and two separate exception classes. `requests` does the same work in `get()` plus `raise_for_status()`. |
| How did each report success? | `urllib` exposes `response.status` and raises `HTTPError` on a bad status. `requests` always returns a response object; you must call `.raise_for_status()` yourself or check `.status_code`. |
| Which on a locked-down computer? | `urllib`, because it is in the standard library and needs no installation permissions. |
| Clearer error message? | `requests`. Its `HTTPError` message includes the status, the reason, and the URL in one readable line. `urllib` gives the code but needs more work to report usefully. |
| Identical files? | Yes, byte for byte. The transport tool cannot alter the content, which is why the checksum in Part 4 is a property of the data and not of how it was fetched. |

**The deliberate failure.** Changing `life-expectancy.csv` to `life-expectancy-typo.csv` produces:

```text
RuntimeError: The server refused https://ourworldindata.org/grapher/life-expectancy-typo.csv
with HTTP status 404
```

raised by the `except HTTPError` branch of `download_with_urllib`. Note that the server answered - it was reachable and it said no. Breaking the *host* instead, to `ourworldindata.invalid`, goes down the `except URLError` branch with a name-resolution error, because no server was ever contacted.

> **Marking note.** Award the marks for a student who names *which branch of their own function* raised the error. A student who only pasted the traceback has demonstrated that an error occurred, not that they understood it.

## Part 4 - Record where the data came from

**TODOs 16 to 22.**

In [ ]:
def file_fingerprint(path):
    """Return the SHA-256 checksum of a file as a 64-character string."""
    # TODO 16
    digest = hashlib.sha256()
    # TODO 17 - chunked so the function works on a file larger than memory.
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(65536), b""):
            digest.update(chunk)
    # TODO 18
    return digest.hexdigest()


def describe_source(provenance, dataset, checksum):
    """Combine download details, dataset details, and a checksum into one record."""
    # TODO 19
    return {
        "source_name": dataset["name"],
        "source_url": provenance["url"],
        "publisher": dataset["publisher"],
        "documentation": dataset["documentation"],
        "local_file": Path(provenance["path"]).name,
        "retrieved_utc": provenance["retrieved_utc"],
        "http_status": provenance["status"],
        "file_bytes": provenance["bytes"],
        "sha256": checksum,
        "row_meaning": dataset["row_meaning"],
        "downloaded_with": provenance["tool"],
    }

In [ ]:
# TODO 20
checksum = file_fingerprint(RAW_PATH)
print("SHA-256:", checksum)

# TODO 21
source_record = describe_source(urllib_provenance, DATASET, checksum)
print(pd.Series(source_record).to_frame("value").to_string())

# TODO 22 - hashing the same unchanged bytes must give the same digest.
second_pass = file_fingerprint(RAW_PATH)
print("\nSecond calculation identical:", checksum == second_pass)

### Provenance - model answers

1. The first 12 characters will differ from this reference if Our World in Data has republished the file since. That is the point of the exercise, not a problem.
2. **Why twice gives the same answer.** SHA-256 is deterministic: the same input bytes always produce the same digest. The file did not change between the two calls, so the digest could not change. If it had changed, something would have been writing to the file.
3. **Detecting a quiet revision.** The filename and the URL would be identical, so nothing visible would tell you the data had changed. The checksum would differ from the one recorded in the manifest, which tells you the file is not the one your earlier analysis used - and therefore that your earlier numbers may no longer be reproducible from the current source.
4. **What a checksum cannot tell you.** Whether the data is *correct*. It detects change, not accuracy. A dataset full of errors has a perfectly stable checksum, and a corrected dataset has a different one - so a changed checksum is not itself bad news. It only tells you to look.

> **Marking note.** Question 4 is where students reveal whether they understood the concept or memorised the word. "It cannot tell you if the data is accurate or unbiased" earns the mark; "it cannot tell you the file size" does not.

## Part 5 - Load and inspect the raw data

**TODOs 23 to 27.**

In [ ]:
def inspect_dataframe(df, name="dataset"):
    """Print a structured audit of a DataFrame and return a summary dictionary."""
    # TODO 23
    print("=" * 70)
    print(f"AUDIT: {name}")
    print("=" * 70)
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")

    print("Columns, types, and missing values:")
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().sum() / len(df) * 100).round(2),
        "unique": df.nunique(),
    })
    print(summary.to_string())

    duplicate_rows = int(df.duplicated().sum())
    print(f"\nFully duplicated rows: {duplicate_rows:,}")

    numeric = df.select_dtypes("number")
    if not numeric.empty:
        print("\nNumeric summary:")
        print(numeric.describe().round(2).to_string())

    print("\nFirst five rows:")
    print(df.head().to_string(index=False))
    print("=" * 70)

    # TODO 24
    return {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": list(df.columns),
        "total_missing": int(df.isna().sum().sum()),
        "duplicate_rows": duplicate_rows,
    }

In [ ]:
# TODO 25 - read the local file, not the URL. It is already downloaded.
raw_df = pd.read_csv(RAW_PATH)

# TODO 26
raw_summary = inspect_dataframe(raw_df, DATASET["name"])

# TODO 27 - confirm the spellings before relying on them.
country_column = DATASET["country_column"]
print("\nEntities containing 'Ghana':",
      [name for name in raw_df[country_column].unique() if "Ghana" in str(name)])

available = set(raw_df[country_column].unique())
missing_names = [name for name in FOCUS_COUNTRIES if name not in available]
print("Focus countries present:", [n for n in FOCUS_COUNTRIES if n in available])
print("Focus countries NOT found:", missing_names or "none - all spellings correct")

# The aggregate-row problem, visible before cleaning.
aggregates = raw_df[raw_df["Code"].isna()][country_column].unique()
print(f"\nEntities with no ISO code ({len(aggregates)} of them) - these are aggregates,")
print("not countries:", list(aggregates[:12]), "...")

### What have you got? - model answers

1. **One row** is one entity in one year, holding that entity's estimated life expectancy at birth in years. "Entity" rather than "country" matters, because the column also contains continents, income groups, and `World`.
2. **21,565 rows and 4 columns** at the time of writing.
3. **`Code` has about 1,348 missing values.** `Life expectancy` has none. The missing codes are not damage - they are the aggregate rows, and their absence is the only reliable way to identify them.
4. **Plausibility.** The full range runs from roughly 17 years to over 85. The low end is real but belongs to famine and war years in individual countries in the nineteenth century, not to any modern country. The year range starting at **1543** is the giveaway that this file blends modern measurement with historical reconstruction, which is why the solution filters to 1960 onwards.
5. **The column that cannot be understood from its name** is `Life expectancy`. It does not say *at what age*, in what *units*, or whether it is period or cohort life expectancy. The Our World in Data page states it is **period life expectancy at birth, in years** - a synthetic measure of what a newborn would live to if current mortality rates held for their whole life. It is not a prediction of how long anyone will actually live. That distinction has to come from the documentation; the numbers alone will never yield it.

> **Marking note - Part 5.** Question 5 is the one worth reading carefully. Students who wrote "Code, because it is not obvious it means ISO country code" get partial credit - it is true and much less consequential. The full marks belong to whoever noticed that "life expectancy" is a technical term they had been reading as plain English. Question 4 rewards anyone who noticed the 1543 start date and asked what it implied.

## Part 6 - Clean the data

**TODOs 28 to 38.**

In [ ]:
COLUMN_RENAMES = {
    "A": {"country": "country", "year": "year", "pop": "population",
          "continent": "continent", "lifeExp": "life_expectancy_years",
          "gdpPercap": "gdp_per_capita_usd"},
    "B": {"Entity": "country", "Code": "country_code", "Year": "year",
          "Life expectancy": "life_expectancy_years"},
    "C": {"Country Name": "country", "Country Code": "country_code",
          "Year": "year", "Value": "gdp_current_usd"},
}


def clean_data(df, countries, start_year, end_year, renames):
    """Return a cleaned copy of the raw data, ready for analysis."""
    df = df.copy()

    # TODO 28
    df = df.rename(columns=renames)

    # TODO 29
    df["year"] = df["year"].astype(int)
    df["life_expectancy_years"] = df["life_expectancy_years"].astype(float)

    # TODO 30
    df = df[df["country"].isin(countries)]
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    # TODO 31 - aggregate rows such as "Africa" and "World" have no ISO code.
    # Filtering to named countries above already excludes them, but the check is
    # kept because it documents the rule and protects the function if somebody
    # later passes a list that accidentally includes an aggregate.
    df = df[df["country_code"].notna()]

    # TODO 32 - Missing values: dropped, not filled. A missing life expectancy
    # is an absent estimate, and inventing one would put a fabricated point on a
    # chart that a reader would take as an observation. Filling with 0 would be
    # worse still: it would claim newborns died immediately and would drag every
    # average down. In this extract there are none, so this drops nothing - but
    # the rule has to be stated for the years and countries where there are.
    df = df.dropna(subset=["life_expectancy_years"])

    # TODO 33
    df = df.drop_duplicates()

    # TODO 34 - years gained since each country's own first year in range.
    df = df.sort_values(["country", "year"])
    df["years_gained_since_start"] = (
        df["life_expectancy_years"]
        - df.groupby("country")["life_expectancy_years"].transform("first")
    ).round(2)

    # TODO 35
    return df.sort_values(["country", "year"]).reset_index(drop=True)

In [ ]:
# TODO 36
clean_df = clean_data(raw_df, FOCUS_COUNTRIES, START_YEAR, END_YEAR,
                      COLUMN_RENAMES[CHOSEN_OPTION])

# TODO 37
clean_summary = inspect_dataframe(clean_df, "cleaned data")

# TODO 38 - each check printed as an explicit True or False.
checks = {
    "row count went down":
        clean_summary["rows"] < raw_summary["rows"],
    "column names are lowercase with no spaces":
        all(col == col.lower() and " " not in col for col in clean_df.columns),
    "year is an integer type":
        pd.api.types.is_integer_dtype(clean_df["year"]),
    "every country is a focus country":
        set(clean_df["country"]) <= set(FOCUS_COUNTRIES),
    "years are within range":
        clean_df["year"].between(START_YEAR, END_YEAR).all(),
    "no missing values in the measure":
        not clean_df["life_expectancy_years"].isna().any(),
}
print("\nCleaning checks:")
for label, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {label}")
print(f"\n{raw_summary['rows']:,} raw rows -> {clean_summary['rows']:,} clean rows")
print(f"Countries: {sorted(clean_df['country'].unique())}")
print(f"Years:     {clean_df['year'].min()} to {clean_df['year'].max()}")

### Cleaning decisions - model answer

| Decision | What I did | Why |
|---|---|---|
| Column names | `Entity`→`country`, `Code`→`country_code`, `Year`→`year`, `Life expectancy`→`life_expectancy_years` | Spaces and capitals force bracket notation and invite typos; the new name carries the unit, so no later reader has to guess whether the figure is in years or months |
| Data types | `year` to `int`, measure to `float` | An integer year sorts and filters correctly and prints without a decimal point on the axis |
| Rows removed | Kept only the three focus countries and 1960-2023; required a non-null `country_code` | Pre-1950 figures are historical reconstructions rather than measurements, and the aggregate rows are not countries, so leaving either in would mean comparing unlike things |
| Missing values | Dropped rather than filled | An absent estimate is not a measurement; filling it would place invented points on the chart, and filling with zero would corrupt every average |
| Derived column | `years_gained_since_start` | The question is about *change*, and the raw level alone does not show it; a derived column makes the answer directly readable |

- **Rows: 21,565 → 192.** Three countries across 64 years.
- **The decision that would change the chart most:** the 1960 start. Extending back to 1543 would compress the entire modern improvement into the right-hand edge of the chart against a long, noisy, sparsely estimated tail, and the reader would take those early reconstructions for observations of the same kind as the recent data. The choice of start year is a claim about which numbers are comparable.

> **Marking note - Part 6.** Full marks require the *reason*, not the action. "I dropped nulls" is half an answer; "I dropped them rather than filling because an absent estimate is not a measurement" is the whole one. Watch specifically for:
>
> - **`fillna(0)` on the measure.** Treat this as a serious error and say why in feedback - it is the difference between reporting and inventing data.
> - **A missing `.copy()`.** Test it by running their Part 6 cell twice. If the second run gives different results, or if `raw_df` has been altered, they modified the caller's DataFrame.
> - **Aggregates left in.** Only students on Option B or C could hit this. A ranking that includes `World` or `Euro area` alongside countries should lose the marks for TODO 31 even if the code runs.

## Part 7 - Save the cleaned data

**TODOs 39 to 44.**

In [ ]:
def save_clean_data(df, destination):
    """Save a cleaned DataFrame to CSV and verify it can be read back."""
    destination = Path(destination)

    # TODO 39 - index=False, or the row numbers become an "Unnamed: 0" column.
    df.to_csv(destination, index=False)

    # TODO 40 - read it back; this catches an empty or half-written file at once.
    reloaded = pd.read_csv(destination)
    round_trip_ok = reloaded.shape == df.shape

    # TODO 41
    return {
        "path": str(destination),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "bytes": destination.stat().st_size,
        "round_trip_ok": bool(round_trip_ok),
    }

In [ ]:
# TODO 42 - a filename that says what is in the file.
CLEAN_PATH = CLEAN_DIR / "life_expectancy_ghana_kenya_senegal_1960_2023.csv"

# TODO 43
clean_record = save_clean_data(clean_df, CLEAN_PATH)
print(json.dumps(clean_record, indent=2))

# TODO 44
print("\nRound trip succeeded:", clean_record["round_trip_ok"])
print("\nFirst three rows read back from disk:")
print(pd.read_csv(CLEAN_PATH).head(3).to_string(index=False))

## Part 8 - Analyse the cleaned data

**TODOs 45 to 52.**

In [ ]:
def summarise_by_country(df, country_column, value_column):
    """Return one row per country summarising the measure."""
    # TODO 45 - "first" and "last" are only meaningful because clean_data()
    # sorted by country then year.
    summary = df.groupby(country_column, as_index=False).agg(
        observations=(value_column, "count"),
        first_value=(value_column, "first"),
        last_value=(value_column, "last"),
        mean_value=(value_column, "mean"),
        min_value=(value_column, "min"),
        max_value=(value_column, "max"),
    )
    # TODO 46
    summary["change"] = summary["last_value"] - summary["first_value"]
    return (summary.sort_values("last_value", ascending=False)
                   .round(2)
                   .reset_index(drop=True))


def top_n_in_year(df, year, value_column, n=5, year_column="year",
                  country_column="country"):
    """Return the n countries with the highest value in a single year."""
    # TODO 47
    return (df[df[year_column] == year]
            .sort_values(value_column, ascending=False)
            .head(n)[[country_column, value_column]]
            .reset_index(drop=True))


def percent_change(df, country_column, year_column, value_column):
    """Return the percentage change in the measure for each country."""
    # TODO 48
    ordered = df.sort_values([country_column, year_column])
    result = ordered.groupby(country_column, as_index=False).agg(
        first_value=(value_column, "first"),
        last_value=(value_column, "last"),
    )
    # Guard against dividing by zero: an undefined change is NaN, not infinity.
    result["percent_change"] = (
        (result["last_value"] - result["first_value"])
        / result["first_value"].where(result["first_value"] != 0)
        * 100
    ).round(2)
    return (result.sort_values("percent_change", ascending=False)
                  .reset_index(drop=True))

In [ ]:
VALUE_COLUMN = "life_expectancy_years"

# TODO 49
country_summary = summarise_by_country(clean_df, "country", VALUE_COLUMN)
print("Summary by country:")
print(country_summary.to_string(index=False))

# TODO 50 - the year is calculated, never typed, so this still works when the
# publisher adds another year of data.
latest_year = int(clean_df["year"].max())
ranking = top_n_in_year(clean_df, latest_year, VALUE_COLUMN, n=5)
print(f"\nRanking in {latest_year}:")
print(ranking.to_string(index=False))

# TODO 51
change_table = percent_change(clean_df, "country", "year", VALUE_COLUMN)
print("\nPercentage change over the period:")
print(change_table.to_string(index=False))

# TODO 52
best = change_table.iloc[0]
print(f"\n{best['country']} improved the most, rising from "
      f"{best['first_value']:.1f} to {best['last_value']:.1f} years "
      f"({best['percent_change']:.1f}% over {START_YEAR}-{END_YEAR}).")

> **Marking note - Part 8.** Two things to check.
>
> **The hardcoded year.** A student who wrote `top_n_in_year(clean_df, 2023, ...)` gets the right answer today and a silently stale one after the next data release. TODO 50 says to calculate it; deduct if they typed it.
>
> **The division guard in TODO 48.** With life expectancy no first value is ever zero, so a student who omitted the guard still gets correct output. Award the mark only if the guard is present, and explain in feedback that the input that breaks it simply is not in this dataset - which is exactly why the habit matters. Accept any correct approach: `.where()`, `.replace(0, np.nan)`, or an explicit conditional.

## Part 9 - Two charts

**TODOs 53 to 62.**

In [ ]:
def plot_line_trend(df, country_column, year_column, value_column,
                    value_label, title, source_note, save_path=None):
    """Draw a line chart of the measure over time, one line per country."""
    fig, ax = plt.subplots()

    # TODO 53 - sort within each group; an unsorted line chart zig-zags.
    for country, group in df.groupby(country_column):
        group = group.sort_values(year_column)
        ax.plot(group[year_column], group[value_column],
                linewidth=2, label=country)

    # TODO 54
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel(value_label)
    ax.legend(title="Country")

    # TODO 55
    ax.grid(True, alpha=0.3)
    fig.text(0.99, -0.03, source_note, ha="right", fontsize=8, color="grey")
    fig.tight_layout()

    # TODO 56
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    # TODO 57
    return ax


def plot_bar_ranking(df, country_column, value_column, value_label, title,
                     source_note, save_path=None):
    """Draw a bar chart ranking countries by the measure."""
    fig, ax = plt.subplots()

    # TODO 58 - label every bar; a reader should not estimate from a height.
    bars = ax.bar(df[country_column], df[value_column],
                  color="#2980b9", edgecolor="white")
    ax.bar_label(bars, fmt="%.1f", fontsize=9, padding=2)

    # TODO 59
    ax.set_title(title)
    ax.set_xlabel("Country")
    ax.set_ylabel(value_label)
    if df[country_column].str.len().max() > 10:
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    fig.text(0.99, -0.03, source_note, ha="right", fontsize=8, color="grey")
    fig.tight_layout()

    # TODO 60
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return ax

In [ ]:
retrieved_date = source_record["retrieved_utc"][:10]
SOURCE_NOTE = f"Source: {DATASET['publisher']}, retrieved {retrieved_date}"

# TODO 61 - the title states the finding, and is checked against the numbers in
# country_summary rather than assumed.
plot_line_trend(
    clean_df, "country", "year", VALUE_COLUMN,
    value_label="Life expectancy at birth (years)",
    # The title states the finding. It was written AFTER reading
    # country_summary, not before, which is why it can name what happened.
    title=(f"Senegal overtook both Ghana and Kenya in life expectancy, "
           f"{START_YEAR}-{END_YEAR}"),
    source_note=SOURCE_NOTE,
    save_path=CHART_DIR / "life_expectancy_trend_line.png",
)
plt.show()

In [ ]:
# TODO 62
plot_bar_ranking(
    ranking, "country", VALUE_COLUMN,
    value_label="Life expectancy at birth (years)",
    title=f"Life expectancy by country in {latest_year}",
    source_note=SOURCE_NOTE,
    save_path=CHART_DIR / "life_expectancy_ranking_bar.png",
)
plt.show()

### Reading the charts - model answers

Figures below are from the run recorded in this notebook. Students' numbers should be close; small differences mean Our World in Data has republished, which is fine and is exactly what the checksum in Part 4 exists to detect.

1. **The line chart** shows all three countries improving over the period, but not in parallel: Senegal rises from the lowest starting point of 39.7 years to the highest final value of 68.7, while Kenya falls from 60.9 years in 1985 to 55.5 in 1997 before recovering to 63.6.
2. **The bar chart** shows the 2023 ranking - Senegal 68.7 years, Ghana 65.5, Kenya 63.6 - a spread of 5.0 years, narrower than the 8.8-year spread of 1960.
3. **Why each chart type.** Year is a continuous ordered variable, so a line correctly implies that the points between the markers exist and that the sequence has direction. Country is a categorical variable with no natural order, so a line between countries would draw a trend where none exists; separate bars compare magnitudes without implying any progression.
4. **The feature worth noticing** is Kenya's dip. It coincides with the peak years of the HIV/AIDS epidemic in East Africa, which raised adult mortality sharply and pulled period life expectancy down before antiretroviral treatment became widely available. Confirming that reading needs cause-of-death or HIV-prevalence data - this dataset shows the effect, not the cause, and a student should say so rather than assert the explanation as established from this file alone.

> **Marking note - Part 9.** The commonest lost marks are for **units** and for a **title that describes the mechanics** rather than the finding: "Line chart of life expectancy" instead of "Life expectancy rose in all three countries". Question 4 is the most valuable answer in the notebook - a student who spotted the Kenya dip, proposed an explanation, *and* said what data would be needed to confirm it is doing the actual work of an analyst. A student who asserted the HIV/AIDS explanation as fact from this dataset should be corrected: the inference is reasonable, the evidence here does not establish it.

## Part 11 - Manifest

**TODOs 63 to 65.** (Part 10 is the optional exploration and is not graded.)

In [ ]:
def build_manifest(source_record, clean_record, cleaning_notes, question,
                   countries, start_year, end_year):
    """Assemble a manifest describing this whole gathering exercise."""
    # TODO 63
    return {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "research_question": question,
        "focus_countries": list(countries),
        "period": {"start": start_year, "end": end_year},
        "raw_source": source_record,
        "clean_output": clean_record,
        "cleaning_steps": list(cleaning_notes),
        "tools": {"pandas": pd.__version__, "python": sys.version.split()[0]},
    }

In [ ]:
# TODO 64
cleaning_notes = [
    "Renamed Entity to country, Code to country_code, Year to year, and "
    "'Life expectancy' to life_expectancy_years so that the unit is carried in "
    "the column name.",
    "Cast year to int and life_expectancy_years to float.",
    "Kept only Ghana, Kenya, and Senegal, and only the years 1960 to 2023 "
    "inclusive, because figures before 1950 are historical reconstructions "
    "rather than measurements.",
    "Required a non-null country_code, which excludes regional aggregates such "
    "as 'Africa' and 'World' that share the Entity column with real countries.",
    "Dropped rows with a missing measure rather than filling them, because an "
    "absent estimate is not a measurement and filling it would put invented "
    "points on the chart.",
    "Dropped exact duplicate rows.",
    "Derived years_gained_since_start as each country's value minus its own "
    "earliest value in range, because the question is about change.",
    "Sorted by country then year and reset the index.",
]

# TODO 65
manifest = build_manifest(source_record, clean_record, cleaning_notes,
                          RESEARCH_QUESTION, FOCUS_COUNTRIES,
                          START_YEAR, END_YEAR)
MANIFEST_PATH = DATA_DIR / "manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Wrote {MANIFEST_PATH.name}\n")
print(MANIFEST_PATH.read_text(encoding="utf-8"))

## Part 12 - Findings - model answer

**1. Answer to the research question.** Life expectancy at birth in Ghana rose from 46.9 years in 1960 to 65.5 years in 2023, a gain of 18.6 years or 39.5%. Senegal changed far more dramatically: it began the period the lowest of the three at 39.7 years and finished the highest at 68.7, gaining 29.0 years or 73.2%. Kenya went the other way relative to its peers - highest in 1960 at 48.5 years and lowest in 2023 at 63.6, gaining only 15.2 years - and its path was the least steady, falling from 60.9 years in 1985 to 55.5 in 1997 before recovering. The three countries were 8.8 years apart in 1960 and 5.0 years apart in 2023, so the gap between them narrowed as well as the level rising.

**2. The most surprising thing.** That the ranking inverted completely. Kenya had the longest life expectancy of the three in 1960 and the shortest by 2023, while Senegal moved from last to first. I had expected all three to improve at broadly similar rates and keep their order. Instead Senegal gained 29.0 years and Kenya 15.2 - almost double. The fact that every line rises hides the fact that the three countries were on very different trajectories, which is a good argument for plotting them together rather than reporting three separate national summaries.

**3. Three limitations.**

1. These are **modelled period life expectancy estimates**, not counts of how long real people lived. Period life expectancy assumes today's mortality rates hold for a newborn's entire life, which they will not. This prevents me from claiming that a child born in Ghana in 2023 will live to 65.
2. The series is a **national average**, so it hides variation within each country. It prevents me from saying anything about differences between Accra and the Northern Region, or between richer and poorer households.
3. Figures for the earlier years, and for periods with weak civil registration, are **reconstructions with wide uncertainty** that the file publishes as single numbers with no confidence interval. This prevents me from treating a one-year difference between two countries in 1960 as a real difference.

**4. What I would gather next.** Under-five mortality and HIV prevalence for the same countries and years, from the World Bank Indicators API. Under-five mortality would test whether Senegal's much faster gain came from improved child survival, which is where large life-expectancy gains usually originate. HIV prevalence would test the reading of Kenya's decline between 1985 and 1997. Both are in the same API, so joining them to this table on country and year is straightforward.

**5. Something that went wrong.** My first ranking chart put `Africa` above every country. The cause was that `Entity` mixes aggregates with countries, and I had filtered by year before looking properly at what the entity column contained. I found it because a continent appearing in a country ranking was obviously wrong, and I fixed it by requiring a non-null `country_code`. The general lesson is the one from Part 5: establish what one row represents before filtering, not after.

> **Marking note - Part 12.** Insist on numbers. "Life expectancy improved a lot" is not an answer at this stage of the course. For limitations, the test is whether the student names **the claim the limitation blocks** - that is what the assignment asked for, and it is what separates a real limitation from a ritual disclaimer. Question 5 rewards honesty; a student who says nothing went wrong either did not attempt enough or is not telling you, and either way it is worth a comment.

## Marking summary

| Area | Marks |
|---|---:|
| Question and dataset choice | 8 |
| Download functions | 16 |
| Comparing urllib and requests | 6 |
| Provenance and checksum | 10 |
| Inspection | 10 |
| Cleaning | 16 |
| Saving and raw/clean separation | 6 |
| Analysis functions | 10 |
| Charts | 12 |
| Manifest | 4 |
| Findings and limitations | 8 |
| Repository, README, and commits | 8 |
| **Total** | **110** |

### The five failures to watch for

1. **Functions that print instead of returning.** Everything downstream breaks, and students often patch around it with globals rather than fixing the function. Fix the function in feedback.
2. **`fillna(0)` on a measurement.** Treat as a serious error, not a style point.
3. **A missing `.copy()` in `clean_data()`.** Test by running the cell twice.
4. **Aggregate rows surviving into a country ranking.** Option B and C only.
5. **A hardcoded latest year.** Correct today, wrong after the next release.

### Repository checks

Open the student's repository and confirm: it is public; `data/raw/` holds the unmodified download; `data/clean/` holds the cleaned file; the notebook shows outputs and charts; `README.md` is complete; commit messages describe changes; and no `.venv` or credentials were committed. A notebook with no visible outputs was pushed without being run, and cannot be marked on its results.